# Objetivo del Notebook
#### Este notebook implementa un pipeline completo de preprocesamiento de datos operacionales de transformadores eléctricos para su uso en sistemas de mantenimiento predictivo. El enfoque está diseñado específicamente para las características técnicas y operacionales de transformadores de potencia, considerando las normativas IEC y las mejores prácticas de la industria eléctrica.

### Tareas Principales
Carga y Consolidación Inteligente: Integración de múltiples archivos de sensores de transformadores
Limpieza Especializada: Tratamiento específico para variables eléctricas y térmicas
Conversión Temporal Avanzada: Manejo de series temporales de monitoreo continuo
Detección de Anomalías: Identificación de estados anómalos basada en criterios técnicos
Etiquetado de Estados: Clasificación graduada (Normal/Alerta/Crítico)
Validación de Calidad: Métricas específicas para datos de transformadores
### Fundamento Técnico
Los transformadores eléctricos son activos críticos en sistemas de potencia que requieren monitoreo continuo de múltiples variables operacionales. Las fallas más comunes incluyen:

Degradación térmica: Sobrecalentamiento del aceite y devanados
Anomalías eléctricas: Desbalances de corriente, sobrecarga, problemas de aislamiento
Fallas mecánicas: Problemas en el cambiador de taps bajo carga (OLTC)
Degradación química: Deterioro del aceite dieléctrico
El preprocesamiento debe preservar las características operacionales críticas mientras elimina ruido y valores espurios típicos de sistemas industriales.

In [1]:
# Importación de librerías esenciales para análisis de transformadores
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
from datetime import datetime, timedelta
import re
import gc
import sys
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Configuración del entorno
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
plt.style.use('default')
sns.set_palette("husl")

print("Librerías importadas exitosamente")
print(f"Versión de pandas: {pd.__version__}")
print(f"Versión de numpy: {np.__version__}")
print(f"Configuración optimizada para análisis de transformadores eléctricos")

Librerías importadas exitosamente
Versión de pandas: 2.2.3
Versión de numpy: 2.2.6
Configuración optimizada para análisis de transformadores eléctricos


In [2]:
# === Verificación rápida de tags en Bronze (Delta) ===
from pathlib import Path
import pandas as pd
from deltalake import DeltaTable

# Usa tu ruta ya definida o ponla manualmente:
# from config_silverv2 import RUTA_BRONZE
# bronze_path = RUTA_BRONZE
bronze_path = Path(r"C:\Users\Asus TUF\Desktop\proy_ml\data\capa_bronze\readings_v1")

def check_bronze_tags(bronze_path: Path):
    if not bronze_path.exists():
        print(f"No existe la ruta Bronze: {bronze_path}")
        return []

    dt = DeltaTable(str(bronze_path))

    # 1) Intentar obtener todos los nombres de columnas vía PyArrow (más confiable)
    try:
        pa_all = dt.to_pyarrow_table()
        all_cols = list(pa_all.schema.names)
    except Exception as e:
        print(f"No se pudo leer la tabla Delta: {e}")
        return []

    print(" Columnas detectadas en Bronze:", all_cols)

    if "tag" not in all_cols:
        print(" La tabla no expone columna 'tag'. No se puede listar etiquetas.")
        return []

    # 2) Listar tags únicos (solo columna tag si es posible)
    try:
        pa_tags = dt.to_pyarrow_table(columns=["tag"])
    except TypeError:
        pa_tags = pa_all  # fallback

    pdf_tags = pa_tags.to_pandas()
    tags = (
        pdf_tags["tag"]
        .astype(str)
        .dropna()
        .unique()
        .tolist()
    )

    print(f" Tags encontrados: {len(tags)}")
    print(" Muestra de tags:", tags[:20])

    # 3) Prueba de lectura filtrada para el primer tag (validación extra)
    if tags:
        t = tags[0]
        candidate_cols = [c for c in ["timestamp","ts","tag","value","value_text","value_bool"] if c in all_cols]
        print(f"\n Probando lectura filtrada para tag='{t}' (columnas: {candidate_cols})")
        try:
            pa_one = dt.to_pyarrow_table(columns=candidate_cols, filters=[("tag","=",t)])
        except TypeError:
            # Algunas versiones no soportan 'filters' → filtrar luego
            pa_one = dt.to_pyarrow_table(columns=candidate_cols)
        pdf_one = pa_one.to_pandas()
        if "tag" in pdf_one.columns:
            pdf_one = pdf_one[pdf_one["tag"] == t]
        print(pdf_one.head())

    return tags

# Ejecutar
_ = check_bronze_tags(bronze_path)


 Columnas detectadas en Bronze: ['timestamp', 'value', 'value_text', 'value_bool', 'tag', 'date']
 Tags encontrados: 91
 Muestra de tags: ['ventilador_14_mcb', 'ventilador_13_mcb', 'ventilador_12_mcb', 'ventilador_11_mcb', 'ventilador_10_mcb', 'ventilador_9_mcb', 'ventilador_8_mcb', 'ventilador_7_mcb', 'ventilador_6_mcb', 'ventilador_5_mcb', 'ventilador_4_mcb', 'ventilador_3_mcb', 'ventilador_2_mcb', 'ventilador_2', 'ventilador_1_mcb', 'ventilador_1', 'resistencia_termica', 'tasa_envejecimiento_30d', 'tasa_envejecimiento', 'consumo_vida_util_ultimo_anio']

 Probando lectura filtrada para tag='ventilador_14_mcb' (columnas: ['timestamp', 'tag', 'value', 'value_text', 'value_bool'])
                         timestamp                tag  value value_text value_bool
0 2024-09-13 05:56:41.676010+00:00  ventilador_14_mcb    NaN       None       None
1 2024-09-13 13:56:41.676010+00:00  ventilador_14_mcb    NaN       None       None
2 2024-09-13 20:03:42.774002+00:00  ventilador_14_mcb    NaN  

In [4]:

def find_repo_root() -> Path:
    p = Path.cwd().resolve()
    for cand in (p, *p.parents):
        if (cand / "data").exists():
            return cand
    return p  # fallback: cwd

# --- Raíz del proyecto ---
BASE_DIR = find_repo_root()

# --- Rutas principales ---
RUTA_RAW      = BASE_DIR / "data" / "raw" / "datos"
RUTA_BRONZE   = BASE_DIR / "data" / "capa_bronze" / "readings_v1"   # Delta Lake
RUTA_SILVER   = BASE_DIR / "data" / "capa_silver" / "preprocesamiento_silver"  # salida de pruebas
RUTA_PROCESSED= BASE_DIR / "data" / "processed"
RUTA_INTERIM  = BASE_DIR / "data" / "interim"
RUTA_REPORTS  = BASE_DIR / "reports"

# --- Crear directorios de salida si no existen (no tocamos Bronze) ---
for ruta in [RUTA_SILVER, RUTA_PROCESSED, RUTA_INTERIM, RUTA_REPORTS]:
    ruta.mkdir(parents=True, exist_ok=True)

# --- Resumen de configuración ---
print(" Configuración de rutas para análisis de transformadores (Notebook)")
print(f"  Base dir:        {BASE_DIR}")
print(f"  Bronze (Delta):  {RUTA_BRONZE}    - {'✅ Existe' if RUTA_BRONZE.exists() else '❌ No existe'}")
print(f"  Silver (prueba): {RUTA_SILVER}    - {'✅ Existe' if RUTA_SILVER.exists() else '❌ No existe'}")
print(f"  Processed:       {RUTA_PROCESSED} - {'✅ Existe' if RUTA_PROCESSED.exists() else '❌ No existe'}")
print(f"  Interim:         {RUTA_INTERIM}   - {'✅ Existe' if RUTA_INTERIM.exists() else '❌ No existe'}")
print(f"  Reports:         {RUTA_REPORTS}   - {'✅ Existe' if RUTA_REPORTS.exists() else '❌ No existe'}")


# --- Inspección rápida de Bronze (Delta) si existe ---
if RUTA_BRONZE.exists():
    print("\n Inspección rápida de Bronze (Delta):")
    delta_log = RUTA_BRONZE / "_delta_log"
    parts = list(RUTA_BRONZE.glob("*.parquet"))
    print(f"   • _delta_log/: {'✅' if delta_log.exists() else '❌'}")
    if delta_log.exists():
        commits = sorted(delta_log.glob("*.json"))[-5:]  # últimos 5
        if commits:
            print("   • Últimos commits (json):")
            for c in commits:
                print(f"     - {c.name}")
    print(f"   • Particiones parquet en raíz: {len(parts)} (muestra hasta 5)")
    for f in parts[:5]:
        tamaño_mb = f.stat().st_size / (1024 * 1024)
        print(f"     - {f.name}: {tamaño_mb:.1f} MB")
else:
    print("\n Bronze (Delta) no disponible en la ruta esperada.")



 Configuración de rutas para análisis de transformadores (Notebook)
  Base dir:        C:\Users\Asus TUF\Desktop\proy_ml
  Bronze (Delta):  C:\Users\Asus TUF\Desktop\proy_ml\data\capa_bronze\readings_v1    - ✅ Existe
  Silver (prueba): C:\Users\Asus TUF\Desktop\proy_ml\data\capa_silver\preprocesamiento_silver    - ✅ Existe
  Processed:       C:\Users\Asus TUF\Desktop\proy_ml\data\processed - ✅ Existe
  Interim:         C:\Users\Asus TUF\Desktop\proy_ml\data\interim   - ✅ Existe
  Reports:         C:\Users\Asus TUF\Desktop\proy_ml\reports   - ✅ Existe

 Inspección rápida de Bronze (Delta):
   • _delta_log/: ✅
   • Últimos commits (json):
     - 00000000000000000086.json
     - 00000000000000000087.json
     - 00000000000000000088.json
     - 00000000000000000089.json
     - 00000000000000000090.json
   • Particiones parquet en raíz: 0 (muestra hasta 5)


In [5]:
# ==================== CARGA DESDE RAW o BRONZE (adaptado) ====================
from pathlib import Path
import pandas as pd
import numpy as np

def cargar_datos_transformador(ruta_datos: Path, fuente: str = "auto"):
    """
    Carga y consolida datos de transformadores desde:
      - RAW: carpeta con '<variable>.parquet'
      - BRONZE (Delta): carpeta de tabla Delta con columnas [timestamp/ts, tag, value/value_text/value_bool]
    
    Params
    ------
    ruta_datos : Path
        Ruta al directorio RAW o a la tabla Delta (Bronze).
    fuente : {"auto","raw","bronze"}
        Modo de lectura. "auto" detecta por presencia de '_delta_log'.

    Returns
    -------
    datos_cargados : dict[str, pd.DataFrame]
        {variable: DataFrame indexado por timestamp (UTC), col = '<variable>_value'}
    archivos_exitosos : list[str]
    archivos_fallidos : list[str]
    """
    print(" Iniciando carga de datos de transformador eléctrico...")

    # Variables esperadas (como en tu código original)
    archivos_transformador = {
        'corriente_carga': 'Corriente de carga por fase',
        'potencia_aparente': 'Potencia aparente total',
        'tap_position': 'Posición del cambiador de taps (OLTC)',
        'temperatura_aceite': 'Temperatura del aceite dieléctrico',
        'temperatura_aceite_OLTC': 'Temperatura del aceite del OLTC',
        'temperatura_ambiente': 'Temperatura ambiente',
        'temperatura_burbujeo': 'Temperatura de burbujeo (indicador gases)',
        'temperatura_punto_caliente': 'Temperatura del punto caliente (hot-spot)',
        'voltaje': 'Voltaje por fases (primario y secundario)'
    }

    datos_cargados, archivos_exitosos, archivos_fallidos = {}, [], []

    ruta_datos = Path(ruta_datos)
    is_bronze_like = (ruta_datos / "_delta_log").exists()
    modo = fuente.lower()
    if modo == "auto":
        modo = "bronze" if is_bronze_like else "raw"

    print(f" Fuente seleccionada: {modo.upper()}  (ruta: {ruta_datos})")

    # ---------- Helpers comunes ----------
    def _finalize(df: pd.DataFrame, var: str, tscol: str | None) -> pd.DataFrame:
        """Normaliza: timestamp→UTC index, renombra valor a '<var>_value', dedup y orden temporal."""
        if tscol is not None:
            df[tscol] = pd.to_datetime(df[tscol], errors="coerce", utc=True)
            df = df.dropna(subset=[tscol]).set_index(tscol)
        else:
            # Si no vino columna temporal, aborta
            raise ValueError("No se encontró columna temporal (timestamp/ts).")

        # Asegurar columna de valor
        valcol = f"{var}_value"
        if "value" in df.columns:
            df = df.rename(columns={"value": valcol})
        elif valcol not in df.columns:
            # Tomar primera numérica como valor si no existe 'value'
            num_cols = [c for c in df.columns if c != tscol and pd.api.types.is_numeric_dtype(df[c])]
            if num_cols:
                df = df.rename(columns={num_cols[0]: valcol})
            else:
                raise ValueError(f"No se encontró columna de valor para '{var}'.")

        df[valcol] = pd.to_numeric(df[valcol], errors="coerce")
        df = df[[valcol]].sort_index()
        # Quitar timestamps duplicados (conserva el último)
        df = df[~df.index.duplicated(keep="last")]
        return df

    # ---------- Lectores por fuente ----------
    def _load_raw(var: str) -> pd.DataFrame:
        fpath = ruta_datos / f"{var}.parquet"
        if not fpath.exists():
            raise FileNotFoundError(f"Archivo no encontrado: {fpath.name}")
        df = pd.read_parquet(fpath)

        # Detectar columna temporal
        cols_time = [c for c in df.columns if any(t in str(c).lower() for t in ["time","fecha","timestamp","hora","ts"])]
        tscol = cols_time[0] if cols_time else None
        return _finalize(df, var, tscol)

    def _load_bronze(var: str) -> pd.DataFrame:
        # Lectura como en tu patrón que sí funcionaba
        from deltalake import DeltaTable
        dt = DeltaTable(str(ruta_datos))

        # detectar columnas disponibles
        try:
            schema_names = dt.schema().names
        except Exception:
            try:
                schema_names = [f.name for f in getattr(dt.schema(), "fields", [])]
            except Exception:
                schema_names = []
        candidate_cols = [c for c in ["timestamp","ts","tag","value","value_text","value_bool"] if c in schema_names]

        # scan con filtro por tag si se puede
        if candidate_cols:
            try:
                pa = dt.to_pyarrow_table(columns=candidate_cols, filters=[("tag","=",var)])
            except TypeError:
                pa = dt.to_pyarrow_table(columns=candidate_cols)
        else:
            pa = dt.to_pyarrow_table()

        pdf = pa.to_pandas()
        if "tag" in pdf.columns:
            pdf = pdf[pdf["tag"] == var].copy()
        if pdf.empty:
            raise ValueError(f"Sin filas para tag='{var}'")

        # Normalizar timestamp (preferir 'timestamp' luego 'ts')
        tscol = "timestamp" if "timestamp" in pdf.columns else ("ts" if "ts" in pdf.columns else None)

        # Unificar columna de valor: value > value_text(num) > value_bool
        if "value" not in pdf.columns:
            pdf["value"] = np.nan
        if pdf["value"].notna().sum() == 0 and "value_text" in pdf.columns:
            maybe = pd.to_numeric(pdf["value_text"], errors="coerce")
            if maybe.notna().sum() > 0:
                pdf["value"] = maybe
        if pdf["value"].notna().sum() == 0 and "value_bool" in pdf.columns:
            pdf["value"] = pdf["value_bool"].astype("Int64")

        return _finalize(pdf, var, tscol)

    # ---------- Loop de carga ----------
    for nombre_var, descripcion in archivos_transformador.items():
        try:
            if modo == "bronze":
                df = _load_bronze(nombre_var)
                #print(df.head(3))  # debug
                print(df.head(3))  # debug
            else:
                df = _load_raw(nombre_var)

            print(f" {nombre_var}: {df.shape[0]:,} registros, 1 columna  —  {descripcion}")
            datos_cargados[nombre_var] = df
            archivos_exitosos.append(nombre_var)

        except Exception as e:
            print(f"  {nombre_var}: fallo de carga → {e}")
            archivos_fallidos.append(nombre_var)

    # ---------- Resumen ----------
    print(f"\n Resumen de carga:")
    print(f"    Variables exitosas: {len(archivos_exitosos)}")
    print(f"    Variables fallidas: {len(archivos_fallidos)}")

    return datos_cargados, archivos_exitosos, archivos_fallidos

# 2) Desde BRONZE (Delta)
datos_transformador, exitosos, fallidos = cargar_datos_transformador(RUTA_BRONZE, fuente="bronze")



 Iniciando carga de datos de transformador eléctrico...
 Fuente seleccionada: BRONZE  (ruta: C:\Users\Asus TUF\Desktop\proy_ml\data\capa_bronze\readings_v1)
                                  corriente_carga_value
timestamp                                              
2024-09-10 00:00:02.080001+00:00             938.154907
2024-09-10 00:00:07.111007+00:00             941.718140
2024-09-10 00:00:12.033004+00:00             939.435669
 corriente_carga: 5,200,026 registros, 1 columna  —  Corriente de carga por fase
                           potencia_aparente_value
timestamp                                         
2024-09-10 04:00:00+00:00                30.917398
2024-09-10 04:01:00+00:00                31.090598
2024-09-10 04:02:00+00:00                30.933044
 potencia_aparente: 377,354 registros, 1 columna  —  Potencia aparente total
                                  tap_position_value
timestamp                                           
2024-09-10 01:59:12.914001+00:00            

In [6]:
def consolidar_datos_transformador(datos_dict):
    """
    Consolida datos de transformador con alineación temporal inteligente
    
    Aplica estrategias específicas según el tipo de variable:
    - Variables térmicas: interpolación suave
    - Variables eléctricas: promedio/muestreo
    - Variables mecánicas: forward-fill
    """
    
    if not datos_dict:
        raise ValueError("No se encontraron datos válidos para consolidar")
    
    print(" Consolidando datos de transformador con alineación temporal...")
    
    # Encontrar el rango temporal común
    indices_temporales = [df.index for df in datos_dict.values() if isinstance(df.index, pd.DatetimeIndex)]
    
    if not indices_temporales:
        print("No se encontraron índices temporales válidos, usando consolidación simple")
        return pd.concat(datos_dict.values(), axis=1, ignore_index=False)
    
    # CORRECCIÓN: Usar un rango temporal más conservador para asegurar solapamiento
    inicio_comun = max(idx.min() for idx in indices_temporales)
    fin_comun = min(idx.max() for idx in indices_temporales)
    
    # Redondear timestamps para evitar problemas de precisión de nanosegundos
    inicio_comun = inicio_comun.round('H')  # Redondear a horas
    fin_comun = fin_comun.round('H')
    
    # CORRECCIÓN: Usar un rango más pequeño para testing inicial (30 días)
    if (fin_comun - inicio_comun).days > 30:
        fin_comun = inicio_comun + pd.Timedelta(days=351)
        print(f"Limitando rango temporal a 30 días para optimizar procesamiento")
    
    print(f"Rango temporal común: {inicio_comun} a {fin_comun}")
    print(f"Duración total: {fin_comun - inicio_comun}")
    
    # Crear índice temporal maestro (resolución horaria para transformadores)
    indice_maestro = pd.date_range(
        start=inicio_comun,
        end=fin_comun,
        freq='H'  # Frecuencia horaria, típica para monitoreo de transformadores
    )
    
    print(f"Índice maestro creado: {len(indice_maestro)} puntos temporales")
    
    # Consolidar cada variable según su tipo
    datos_consolidados = pd.DataFrame(index=indice_maestro)
    
    # Categorización de variables según su naturaleza física
    variables_termicas = ['temperatura_aceite', 'temperatura_punto_caliente', 
                         'temperatura_ambiente', 'temperatura_burbujeo', 'temperatura_aceite_OLTC']
    variables_electricas = ['corriente_carga', 'voltaje', 'potencia_aparente']
    variables_mecanicas = ['tap_position']
    
    for nombre_var, df in datos_dict.items():
        if not isinstance(df.index, pd.DatetimeIndex):
            print(f" Saltando {nombre_var}: no tiene índice temporal")
            continue
        
        # CORRECCIÓN: Filtrar al rango común usando máscara booleana más robusta
        mask_rango = (df.index >= inicio_comun) & (df.index <= fin_comun)
        df_filtrado = df.loc[mask_rango]
        
        if df_filtrado.empty:
            print(f" Saltando {nombre_var}: sin datos en rango común")
            continue
            
        print(f"   {nombre_var}: {len(df_filtrado)} registros en rango temporal")
        
        try:
            # Aplicar estrategia de consolidación según tipo de variable
            if nombre_var in variables_termicas:
                # Variables térmicas: interpolación suave (cambios graduales)
                df_resampled = df_filtrado.resample('H').mean().interpolate(method='time')
                metodo = "interpolación térmica"
                
            elif nombre_var in variables_electricas:
                # Variables eléctricas: promedio por hora (captura variabilidad)  
                df_resampled = df_filtrado.resample('H').mean()
                metodo = "promedio eléctrico"
                
            elif nombre_var in variables_mecanicas:
                # Variables mecánicas: último valor válido (cambios discretos)
                df_resampled = df_filtrado.resample('H').last().fillna(method='ffill')
                metodo = "último valor mecánico"
                
            else:
                # Otras variables: interpolación general
                df_resampled = df_filtrado.resample('H').mean().interpolate(method='linear')
                metodo = "interpolación general"
            
            print(f"   {nombre_var}: remuestreo completado, {df_resampled.notna().sum().sum()} datos válidos")

            # Alinear con índice maestro -  verificar alineación
            df_alineado = df_resampled.reindex(indice_maestro)
            
            # Verificar que la alineación fue exitosa
            datos_alineados_validos = df_alineado.notna().sum().sum()
            print(f"   {nombre_var}: {df_alineado.shape[1]} columnas, {datos_alineados_validos} datos alineados")
            
            # Agregar al dataset consolidado
            datos_consolidados = datos_consolidados.join(df_alineado, how='left')
            
        except Exception as e:
            print(f"    Error procesando {nombre_var}: {str(e)}")
            continue
    
    # Verificación final
    total_datos_validos = datos_consolidados.notna().sum().sum()
    
    print(f"\n Dataset consolidado final:")
    print(f"   Dimensiones: {datos_consolidados.shape[0]:,} × {datos_consolidados.shape[1]}")
    print(f"   Resolución temporal: 1 hora")
    print(f"   Cobertura: {datos_consolidados.index.min()} a {datos_consolidados.index.max()}")
    print(f"   Total datos válidos: {total_datos_validos:,}")

    if total_datos_validos == 0:
        print("    ADVERTENCIA: No hay datos válidos en el dataset consolidado")
        print("    Revisar alineación temporal y rangos de datos")
    
    return datos_consolidados

# Consolidación
df_transformador = consolidar_datos_transformador(datos_transformador)

 Consolidando datos de transformador con alineación temporal...
Limitando rango temporal a 30 días para optimizar procesamiento
Rango temporal común: 2024-09-10 04:00:00+00:00 a 2025-08-27 04:00:00+00:00
Duración total: 351 days 00:00:00
Índice maestro creado: 8425 puntos temporales
   corriente_carga: 5186413 registros en rango temporal
   corriente_carga: remuestreo completado, 8350 datos válidos
   corriente_carga: 1 columnas, 8350 datos alineados
   potencia_aparente: 377304 registros en rango temporal
   potencia_aparente: remuestreo completado, 8220 datos válidos
   potencia_aparente: 1 columnas, 8220 datos alineados
   tap_position: 7140 registros en rango temporal
   tap_position: remuestreo completado, 8416 datos válidos
   tap_position: 1 columnas, 8416 datos alineados
   temperatura_aceite: 13403 registros en rango temporal
   temperatura_aceite: remuestreo completado, 8422 datos válidos
   temperatura_aceite: 1 columnas, 8422 datos alineados
   temperatura_aceite_OLTC: 1149

In [7]:
# ==================== CONSOLIDACIÓN TEMPORAL INTELIGENTE (adaptado) ====================
import pandas as pd
import numpy as np

def consolidar_datos_transformador(
    datos_dict: dict[str, pd.DataFrame],
    freq: str = "60T",               # Frecuencia objetivo (Silver suele usar 30 minutos)
    max_days: int | None = None,     # Limitar rango para pruebas (None = sin límite)
    interp_limits: dict[str, int] | None = None,  # límites de pasos por variable (override)
    limit_termicas: int = 8,         # ~4h si freq=30T
    limit_electricas: int = 4,       # ~2h si freq=30T
    limit_mecanicas: int = 8,        # ~4h para ffill/bfill
):
    """
    Consolida datos de transformador con alineación temporal inteligente.
    Estrategias:
      - Térmicas: mean + interpolate('time') con límite de pasos
      - Eléctricas: mean + interpolate('time') más conservadora (límite menor)
      - Mecánicas: last + ffill/bfill con límite (discretas)
    Retorna un DataFrame ancho indexado por el índice maestro (freq) con
    columnas = '<variable>_value'.
    """
    if not datos_dict:
        raise ValueError("No se encontraron datos válidos para consolidar")

    print(" Consolidando datos de transformador con alineación temporal…")

    # --- Categorías por naturaleza física (basado en tus variables) ---
    variables_termicas   = {'temperatura_aceite','temperatura_punto_caliente','temperatura_ambiente',
                            'temperatura_burbujeo','temperatura_aceite_OLTC'}
    variables_electricas = {'corriente_carga','voltaje','potencia_aparente'}
    variables_mecanicas  = {'tap_position'}

    # --- Rango temporal común (intersección) ---
    indices_temporales = [df.index for df in datos_dict.values() if isinstance(df.index, pd.DatetimeIndex)]
    if not indices_temporales:
        print(" No hay índices temporales válidos; concatenación simple.")
        return pd.concat(datos_dict.values(), axis=1, ignore_index=False)

    inicio_comun = max(idx.min() for idx in indices_temporales)
    fin_comun    = min(idx.max() for idx in indices_temporales)

    # Normalizar a bordes exactos de la frecuencia objetivo (evita desalineaciones)
    try:
        # Si freq es de pandas (ej. "30T", "1H"), podemos usar floor/ceil
        inicio_comun = pd.to_datetime(inicio_comun).ceil(freq)
        fin_comun    = pd.to_datetime(fin_comun).floor(freq)
    except Exception:
        # Fallback seguro
        inicio_comun = pd.to_datetime(inicio_comun)
        fin_comun    = pd.to_datetime(fin_comun)

    if max_days and (fin_comun - inicio_comun).days > max_days:
        fin_comun = inicio_comun + pd.Timedelta(days=max_days)
        print(f"Limitando rango temporal a {max_days} días para optimizar procesamiento")

    if fin_comun <= inicio_comun:
        raise ValueError("Rango temporal común vacío; revisa la intersección de fechas entre variables.")

    # Detectar zona horaria (usamos la del primer índice) para crear el índice maestro
    tzinfo = indices_temporales[0].tz
    indice_maestro = pd.date_range(start=inicio_comun, end=fin_comun, freq=freq, tz=tzinfo)
    print(f" Rango común: {inicio_comun} → {fin_comun} | ⏱️ {fin_comun - inicio_comun}")
    print(f" Índice maestro creado: {len(indice_maestro)} puntos ({freq})")

    # --- Contenedor del consolidado ---
    datos_consolidados = pd.DataFrame(index=indice_maestro)

    # --- Funciones auxiliares ---
    def _valcol(df: pd.DataFrame) -> str:
        """Devuelve el nombre de la única columna de valor (p.ej., '<var>_value')."""
        # Usamos la primera columna no vacía; tu loader deja 1 col por df
        return df.columns[0]

    def _limite_var(var: str, por_defecto: int) -> int:
        if interp_limits and var in interp_limits:
            return interp_limits[var]
        return por_defecto

    # --- Bucle por variable ---
    for var, df in datos_dict.items():
        if not isinstance(df.index, pd.DatetimeIndex):
            print(f"Saltando {var}: no tiene índice temporal")
            continue

        # Filtrar al rango común
        df = df.sort_index()
        df = df.loc[(df.index >= inicio_comun) & (df.index <= fin_comun)]
        if df.empty:
            print(f"Saltando {var}: sin datos en rango común")
            continue

        col = _valcol(df)
        # Asegurar numérico blando
        df[col] = pd.to_numeric(df[col], errors="coerce")

        try:
            # --- Estrategias por tipo ---
            if var in variables_termicas:
                # mean + interpolate(time) con límite (cambios graduales)
                lim = _limite_var(var, limit_termicas)
                df_res = df.resample(freq).mean()
                df_res[col] = df_res[col].interpolate(
                    method="time", limit=lim, limit_direction="both"
                )
                metodo = f"térmica (mean + interp time, limit={lim})"

            elif var in variables_electricas:
                # mean + interpolate(time) conservadora (límite menor)
                lim = _limite_var(var, limit_electricas)
                df_res = df.resample(freq).mean()
                df_res[col] = df_res[col].interpolate(
                    method="time", limit=lim, limit_direction="both"
                )
                metodo = f"eléctrica (mean + interp time, limit={lim})"

            elif var in variables_mecanicas:
                # last + ffill/bfill acotado (discreto)
                lim = _limite_var(var, limit_mecanicas)
                df_res = df.resample(freq).last()
                # ffill/bfill con límite de pasos (no inventar grandes tramos)
                df_res[col] = df_res[col].ffill(limit=lim).bfill(limit=lim)
                # Si es tap, intenta entero
                if var == "tap_position":
                    df_res[col] = pd.to_numeric(df_res[col], errors="coerce").round().astype("Int64")
                metodo = f"mecánica (last + ffill/bfill, limit={lim})"

            else:
                # Fallback general: mean + interpolate(linear)
                df_res = df.resample(freq).mean()
                df_res[col] = df_res[col].interpolate(method="linear", limit_direction="both")
                metodo = "general (mean + interp linear)"

            # Reindex exacto al maestro (por si faltan bordes)
            df_aln = df_res.reindex(indice_maestro)

            # Reporte
            validos = int(df_aln[col].notna().sum())
            print(f"    {var:25} → {metodo:42s} | válidos: {validos:7,d}")

            # Join al consolidado (mantiene nombre '<var>_value')
            datos_consolidados = datos_consolidados.join(df_aln[[col]], how="left")

        except Exception as e:
            print(f"    Error procesando {var}: {e}")
            continue

    # --- Verificación final ---
    total_validos = int(datos_consolidados.notna().sum().sum())
    print("\n Dataset consolidado final")
    print(f"   Dimensiones: {datos_consolidados.shape[0]:,} × {datos_consolidados.shape[1]}")
    print(f"   Resolución: {freq}")
    print(f"   total datos válidos: {total_validos:,}")

    if total_validos == 0:
        print("    ADVERTENCIA: No hay datos válidos en el dataset consolidado")
        print("    Revisa la intersección temporal, la frecuencia y los límites de imputación")

    return datos_consolidados


# ==================== EJECUCIÓN ====================
# Usa el dict que cargaste antes con el loader.
df_transformador = consolidar_datos_transformador(
    datos_transformador,
    # freq="30T",         # Silver típico
    # max_days=30,        # útil para pruebas; pon None para todo el rango
    # interp_limits={"temperatura_aceite": 12}  # override opcional por variable
)


 Consolidando datos de transformador con alineación temporal…
 Rango común: 2024-09-10 04:00:00+00:00 → 2025-08-27 21:00:00+00:00 | ⏱️ 351 days 17:00:00
 Índice maestro creado: 8442 puntos (60T)
    corriente_carga           → eléctrica (mean + interp time, limit=4)    | válidos:   8,400
    potencia_aparente         → eléctrica (mean + interp time, limit=4)    | válidos:   8,405
    tap_position              → mecánica (last + ffill/bfill, limit=8)     | válidos:   8,402
    temperatura_aceite        → térmica (mean + interp time, limit=8)      | válidos:   8,408
    temperatura_aceite_OLTC   → térmica (mean + interp time, limit=8)      | válidos:   8,407
    temperatura_ambiente      → térmica (mean + interp time, limit=8)      | válidos:   8,410
    temperatura_burbujeo      → térmica (mean + interp time, limit=8)      | válidos:   8,410
    temperatura_punto_caliente → térmica (mean + interp time, limit=8)      | válidos:   8,408
    voltaje                   → eléctrica (mean + in

In [ ]:
#FRECUENCIA
df_transformador.index.freq = pd.tseries.frequencies.to_offset("30T") #30T es 

<Hour>

In [12]:
# ==================== LIMPIEZA / ESTANDARIZACIÓN DE NOMBRES ====================
import re
import pandas as pd

def limpiar_nombres_columnas_transformador(df: pd.DataFrame) -> pd.DataFrame:
    """
    Limpia y estandariza nombres de columnas específicos para transformadores.
    Estrategia:
      1) Mapeo explícito (castellano → inglés técnico).
      2) Limpieza general (lower, sin acentos, snake_case, tokens técnicos).
      3) Resolución de duplicados.
    No toca el índice (timestamp).
    """
    print(" Limpiando nombres de columnas para transformadores...")

    # Vista previa
    muestra = list(map(str, df.columns[:10]))
    print("\n Muestra de nombres originales:")
    for i, col in enumerate(muestra, 1):
        print(f"   {i}. '{col}'")

    # Columnas "especiales" que no queremos tocar si existen
    passthrough = {
        "estado_operacional", "nivel_severidad", "variables_anomalas", "descripcion_anomalia"
    }

    # 1) Mapeo explícito (consistente con tu pipeline)
    explicit_map = {
        "corriente_carga_value": "current_load_value",
        "potencia_aparente_value": "power_apparent_value",
        "tap_position_value": "tap_position_value",
        "temperatura_aceite_value": "temp_oil_value",
        "temperatura_aceite_oltc_value": "temp_oil_oltc_value",   # normaliza minúsculas
        "temperatura_ambiente_value": "temp_ambient_value",
        "temperatura_burbujeo_value": "temp_bubbling_value",
        "temperatura_punto_caliente_value": "temp_spot_hot_value",
        "voltaje_value": "voltage_value",
    }

    # 2) Reglas léxicas (castellano → inglés técnico)
    tech_tokens = {
        "temperatura": "temp",
        "corriente": "current",
        "voltaje": "voltage",
        "tensión": "voltage",
        "potencia": "power",
        "aparente": "apparent",
        "aceite": "oil",
        "ambiente": "ambient",
        "caliente": "hot",
        "punto": "spot",
        "burbujeo": "bubbling",
        "posicion": "position",
        "posición": "position",
        "carga": "load",
    }

    # Utilidad: quitar acentos (rápido y local)
    def _strip_accents(s: str) -> str:
        # mapea: áéíóúñ → aeioun
        return re.sub(r"[áÁ]", "a", re.sub(r"[éÉ]", "e",
               re.sub(r"[íÍ]", "i", re.sub(r"[óÓ]", "o",
               re.sub(r"[úÚ]", "u", re.sub(r"[ñÑ]", "n", s))))))

    # Limpieza general
    def _clean_general(name: str) -> str:
        n = str(name)
        n = n.strip()
        n = _strip_accents(n.lower())

        # reemplazo de tokens técnicos
        for es, en in tech_tokens.items():
            n = n.replace(es, en)

        # snake_case básico y caracteres válidos
        n = re.sub(r"\s+", "_", n)
        n = re.sub(r"[^a-z0-9_]", "", n)
        n = re.sub(r"_+", "_", n).strip("_")

        # si quedó vacío, crea un placeholder
        if not n:
            n = "var_transformador"
        return n

    # Construir nuevos nombres con preferencia por el mapeo explícito
    new_names = []
    applied_map = {}  # para mostrar qué se renombró a qué
    for col in df.columns:
        col_str = str(col)

        # Si es passthrough, lo dejamos igual
        if col_str in passthrough:
            new = col_str

        # Si ya coincide con el explícito como destino, respeta (idempotente)
        elif col_str in explicit_map.values():
            new = col_str

        # Si existe en el mapeo fuente
        elif col_str in explicit_map:
            new = explicit_map[col_str]
            applied_map[col_str] = new

        else:
            # Limpieza general
            new = _clean_general(col_str)
            # Si no contiene sufijo 'value' y parece una serie (muy común en tu flujo), lo mantenemos tal cual;
            # si quieres forzar sufijo, podrías añadir aquí una regla.
            if new.endswith("_value") is False and col_str.endswith("_value"):
                # Conserva la intención original del sufijo si venía en el nombre original
                new = f"{new}_value"
            if new != col_str:
                applied_map[col_str] = new

        new_names.append(new)

    # Resolver duplicados
    final_names = []
    seen = {}
    for name in new_names:
        if name in seen:
            seen[name] += 1
            final = f"{name}_{seen[name]}"
        else:
            seen[name] = 0
            final = name
        final_names.append(final)

    # Aplicar
    df = df.copy()
    df.columns = final_names

    # Vista previa final
    print("\nMuestra de nombres limpios :")
    for i, col in enumerate(df.columns[:10], 1):
        print(f"   {i}. '{col}'")

    # Mapeo aplicado (para trazabilidad en logs del notebook)
    if applied_map:
        print("\n Mapeo aplicado (origen → destino):")
        for k, v in applied_map.items():
            print(f"   • {k} → {v}")

    print(f"\n Limpieza de nombres completada: {len(final_names)} columnas procesadas")
    return df


# ========= APLICAR =========
df_clean = limpiar_nombres_columnas_transformador(df_transformador.copy())


 Limpiando nombres de columnas para transformadores...

 Muestra de nombres originales:
   1. 'corriente_carga_value'
   2. 'potencia_aparente_value'
   3. 'tap_position_value'
   4. 'temperatura_aceite_value'
   5. 'temperatura_aceite_OLTC_value'
   6. 'temperatura_ambiente_value'
   7. 'temperatura_burbujeo_value'
   8. 'temperatura_punto_caliente_value'
   9. 'voltaje_value'

Muestra de nombres limpios :
   1. 'current_load_value'
   2. 'power_apparent_value'
   3. 'tap_position_value'
   4. 'temp_oil_value'
   5. 'temp_oil_oltc_value'
   6. 'temp_ambient_value'
   7. 'temp_bubbling_value'
   8. 'temp_spot_hot_value'
   9. 'voltage_value'

 Mapeo aplicado (origen → destino):
   • corriente_carga_value → current_load_value
   • potencia_aparente_value → power_apparent_value
   • temperatura_aceite_value → temp_oil_value
   • temperatura_aceite_OLTC_value → temp_oil_oltc_value
   • temperatura_ambiente_value → temp_ambient_value
   • temperatura_burbujeo_value → temp_bubbling_value
  

In [13]:
# ==================== CONVERSIÓN DE TIPOS + VALIDACIÓN TÉCNICA (adaptado) ====================
from typing import Dict, Tuple, Iterable

def convertir_tipos_transformador(
    df: pd.DataFrame,
    policy: str = "nan",                      # 'nan' | 'clip' | 'ignore'
    rangos_override: Dict[str, Tuple[float, float]] | None = None,
    entero_cols: Iterable[str] = ("tap_position", "tap_position_value"),
    min_success: float = 0.70,               # mínimo % de valores que deben sobrevivir a la conversión
) -> pd.DataFrame:
    """
    Convierte a numérico e impone validaciones físicas de transformadores.
    
    Reglas base por tokens en el nombre de columna:
      - Temperatura (°C): 'temp' (-50, 200), 'oil' (0, 150), 'hot' (0, 180), 'ambient' (-40, 60)
      - Eléctricas: 'current' (0, 5000 A), 'voltage' (0, 50000 V), 'power' (0, 100000 kVA)
      - Mecánicas: 'position' (0, 20), 'tap' (0, 20)
    
    Parámetros:
      policy:    qué hacer con valores fuera de rango: 'nan' (NaN), 'clip' (saturar al min/max), 'ignore' (dejarlos).
      rangos_override: rangos específicos por *nombre exacto de columna* (tiene prioridad).
      entero_cols: columnas que deben quedar como enteras (Int64) tras redondeo.
      min_success: umbral mínimo de retención (post-conversión) para considerar “exitosa” la conversión.
    """
    assert policy in {"nan", "clip", "ignore"}, "policy debe ser 'nan', 'clip' o 'ignore'"

    print(" Convirtiendo tipos con validación técnica para transformadores...")

    # Info del índice temporal (si aplica)
    if isinstance(df.index, pd.DatetimeIndex):
        print(f" Índice temporal validado: {len(df.index)} registros")
        print(f" Rango: {df.index.min()} a {df.index.max()}")
        print(f" Duración: {df.index.max() - df.index.min()}")

    # Rangos técnicos por token (orden de prioridad importante)
    rangos_tecnicos_tokens: list[tuple[str, tuple[float, float]]] = [
        # El orden define precedencia si una col matchea varios tokens
        ("temp",    (0, 200)),   # genérico temp
        ("oil",     (0, 150)),
        ("hot",     (0, 180)),
        ("ambient", (-40, 60)),
        ("current", (0, 5000)),
        ("voltage", (0, 50000)),
        ("power",   (0, 100000)),
        ("position",(0, 20)),
        ("tap",     (0, 20)),
    ]
    rangos_override = rangos_override or {}

    conversiones_exitosas = 0
    valores_fuera_rango = 0
    valores_clip = 0

    print(f"\n Procesando {len(df.columns)} columnas...")

    df = df.copy()

    def _rango_para_col(col: str) -> Tuple[float, float] | None:
        # 1) override exacto por nombre de columna
        if col in rangos_override:
            return rangos_override[col]
        # 2) por tokens en nombre
        lc = col.lower()
        for token, (a, b) in rangos_tecnicos_tokens:
            if token in lc:
                return (a, b)
        return None

    for col in df.columns:
        serie = df[col]
        # Salta columnas no numéricas que claramente no son series (e.g., etiquetas)
        if pd.api.types.is_datetime64_any_dtype(serie.dtype):
            continue

        try:
            # Conversión a numérico
            n_before = int(serie.notna().sum())
            serie_num = pd.to_numeric(serie, errors="coerce")

            rango = _rango_para_col(col)
            if rango is not None:
                a, b = rango
                mask_validos = serie_num.notna()
                mask_low  = (serie_num < a) & mask_validos
                mask_high = (serie_num > b) & mask_validos
                n_low  = int(mask_low.sum())
                n_high = int(mask_high.sum())
                n_oor  = n_low + n_high

                if n_oor > 0:
                    if policy == "nan":
                        serie_num[mask_low | mask_high] = np.nan
                        valores_fuera_rango += n_oor
                        print(f"   ⚠️ {col}: {n_oor} fuera de rango [{a}, {b}] → NaN")
                    elif policy == "clip":
                        # Saturar
                        serie_num[mask_low]  = a
                        serie_num[mask_high] = b
                        valores_clip += n_oor
                        print(f"   ⚠️ {col}: {n_oor} fuera de rango [{a}, {b}] → clip")
                    else:
                        # ignore
                        print(f"   ⚠️ {col}: {n_oor} fuera de rango [{a}, {b}] (ignorados)")

            n_after = int(serie_num.notna().sum())
            # Éxito mínimo
            if n_before == 0:
                # nada que convertir realmente
                df[col] = serie_num
                print(f"    {col}: sin valores válidos antes/después")
            elif n_after >= int(n_before * min_success):
                # enteros discretos para columnas tipo tap/position
                if any(tok in col.lower() for tok in ["tap", "position"]) or col in entero_cols:
                    serie_num = pd.to_numeric(serie_num, errors="coerce").round().astype("Int64")
                df[col] = serie_num
                conversiones_exitosas += 1
                if rango is not None:
                    print(f"   {col}: convertida (retención {n_after}/{n_before}), rango aplicado {rango}")
                else:
                    print(f"   {col}: convertida (retención {n_after}/{n_before}), sin rango específico")
            else:
                print(f"    {col}: conversión perdió muchos valores ({n_after}/{n_before}) — se conserva sin cambios")

        except Exception as e:
            print(f"    Error en {col}: {e}")

    print(f"\nResumen de conversión:")
    print(f"    Conversiones exitosas: {conversiones_exitosas}/{len(df.columns)}")
    if policy == "nan":
        print(f"    Valores marcados NaN por rango: {valores_fuera_rango:,}")
    elif policy == "clip":
        print(f"    Valores clippeados por rango: {valores_clip:,}")
    else:
        print(f"    Política 'ignore': no se alteraron valores fuera de rango")
    print(f"   Tipos resultantes: {dict(df.dtypes.value_counts())}")

    return df

# ===== Aplicar =====
df_typed = convertir_tipos_transformador(
    df_clean,
    # policy="nan",                # 'nan' (default) | 'clip' | 'ignore'
    # rangos_override=None,        # p.ej., {'voltage_value': (1000, 25000)}
    # entero_cols=("tap_position", "tap_position_value"),
    # min_success=0.70
)


 Convirtiendo tipos con validación técnica para transformadores...
 Índice temporal validado: 8442 registros
 Rango: 2024-09-10 04:00:00+00:00 a 2025-08-27 21:00:00+00:00
 Duración: 351 days 17:00:00

 Procesando 9 columnas...
   current_load_value: convertida (retención 8400/8400), rango aplicado (0, 5000)
   power_apparent_value: convertida (retención 8405/8405), rango aplicado (0, 100000)
   tap_position_value: convertida (retención 8402/8402), rango aplicado (0, 20)
   temp_oil_value: convertida (retención 8408/8408), rango aplicado (0, 200)
   temp_oil_oltc_value: convertida (retención 8407/8407), rango aplicado (0, 200)
   temp_ambient_value: convertida (retención 8410/8410), rango aplicado (0, 200)
   temp_bubbling_value: convertida (retención 8410/8410), rango aplicado (0, 200)
   temp_spot_hot_value: convertida (retención 8408/8408), rango aplicado (0, 200)
   voltage_value: convertida (retención 8400/8400), rango aplicado (0, 50000)

Resumen de conversión:
    Conversiones ex

In [14]:
import pandas as pd
import numpy as np

def analizar_valores_faltantes_transformador_min(df: pd.DataFrame):
    """
    Análisis mínimo de faltantes para Silver:
      - % faltantes por categoría técnica
      - % faltantes por variable (ranking)
      - Períodos críticos simples (>50% columnas NaN simultáneamente), si hay DatetimeIndex
    """
    print("Analizando valores faltantes (versión mínima)…")
    if df.empty:
        print(" DataFrame vacío.")
        return {'categorias': {}, 'variables': pd.DataFrame(), 'periodos_criticos': pd.DataFrame()}

    # 1) Categorías por tokens (fijos)
    categorias = {
        'Térmica':   ['temp', 'oil', 'hot', 'ambient', 'bubbling'],
        'Eléctrica': ['current', 'voltage', 'power', 'apparent'],
        'Mecánica':  ['position', 'tap'],
        'Otras':     []  # se completa al final con lo que no entró arriba
    }

    # 2) % faltantes por variable (ranking)
    registros = len(df)
    vars_rows = []
    for col in df.columns:
        n_miss = int(df[col].isna().sum())
        pct    = (n_miss / registros * 100) if registros else 0.0
        vars_rows.append({'variable': col, 'faltantes': n_miss, 'porcentaje_faltantes': pct})
    df_vars = pd.DataFrame(vars_rows).sort_values('porcentaje_faltantes', ascending=False).reset_index(drop=True)

    # 3) % faltantes por categoría
    cols_categorizadas = set()
    stats_categorias = {}
    for cat, toks in categorias.items():
        if toks:
            cols = [c for c in df.columns if any(t in c.lower() for t in toks)]
        else:
            cols = []
        if cols:
            cols_categorizadas.update(cols)
            miss = int(df[cols].isna().sum().sum())
            total = int(df[cols].size)
            stats_categorias[cat] = {
                'columnas': len(cols),
                'valores_faltantes': miss,
                'porcentaje': (miss / total * 100) if total > 0 else 0.0,
                'columnas_lista': cols
            }

    # completar 'Otras'
    otras = [c for c in df.columns if c not in cols_categorizadas]
    if otras:
        miss = int(df[otras].isna().sum().sum())
        total = int(df[otras].size)
        stats_categorias['Otras'] = {
            'columnas': len(otras),
            'valores_faltantes': miss,
            'porcentaje': (miss / total * 100) if total > 0 else 0.0,
            'columnas_lista': otras
        }

    # 4) Períodos críticos simples (>50% columnas NaN simultáneamente)
    if isinstance(df.index, pd.DatetimeIndex):
        prop_na = df.isna().mean(axis=1)          # proporción [0..1] por timestamp
        mask = prop_na > 0.5                      # umbral fijo 50%
        periodos = []
        if mask.any():
            m = mask.to_numpy()
            # detectar runs contiguos de True
            change = np.diff(m.astype(int), prepend=m[0])
            starts = np.where((change == 1) & m)[0]
            ends   = np.where((change == -1) & (~m))[0] - 1
            if m[-1]: ends = np.append(ends, len(m) - 1)
            if m[0]:
                if len(starts) == 0 or starts[0] != 0:
                    starts = np.insert(starts, 0, 0)
            for s, e in zip(starts, ends):
                ts_slice = df.index[s:e+1]
                periodos.append({
                    'inicio': ts_slice[0],
                    'fin': ts_slice[-1],
                    'n_steps': int(e - s + 1),
                    'pct_faltantes_prom': float(prop_na.iloc[s:e+1].mean() * 100),
                    'pct_faltantes_max': float(prop_na.iloc[s:e+1].max() * 100),
                })
        df_periodos = pd.DataFrame(periodos)
    else:
        df_periodos = pd.DataFrame()

    # 5) Impresiones mínimas
    total_missing = int(df.isna().sum().sum())
    total_vals = int(df.size)
    pct_total = (total_missing / total_vals * 100) if total_vals else 0.0

    print(f" Total valores: {total_vals:,} |  Faltantes: {total_missing:,} ({pct_total:.2f}%)")
    print(" % faltantes por categoría:")
    for cat, s in stats_categorias.items():
        print(f"   - {cat}: {s['porcentaje']:.1f}% ({s['valores_faltantes']:,} / {s['columnas']} cols)")

    print("\n Top variables con más faltantes:")
    print(df_vars[['variable','porcentaje_faltantes','faltantes']].head(10).to_string(index=False))

    if not df_periodos.empty:
        print(f"\n Períodos críticos detectados: {len(df_periodos)} (umbral fijo 50%)")
        print(df_periodos.head(min(5, len(df_periodos))).to_string(index=False))
    else:
        print("\nSin períodos críticos (o índice no temporal).")

    return {'categorias': stats_categorias, 'variables': df_vars, 'periodos_criticos': df_periodos}

# ==== Ejecutar (drop-in replacement) ====
stats_faltantes = analizar_valores_faltantes_transformador_min(df_typed)
        

Analizando valores faltantes (versión mínima)…
 Total valores: 75,978 |  Faltantes: 328 (0.43%)
 % faltantes por categoría:
   - Térmica: 0.4% (167 / 5 cols)
   - Eléctrica: 0.5% (121 / 3 cols)
   - Mecánica: 0.5% (40 / 1 cols)

 Top variables con más faltantes:
            variable  porcentaje_faltantes  faltantes
  current_load_value              0.497512         42
       voltage_value              0.497512         42
  tap_position_value              0.473821         40
power_apparent_value              0.438285         37
 temp_oil_oltc_value              0.414594         35
      temp_oil_value              0.402748         34
 temp_spot_hot_value              0.402748         34
 temp_bubbling_value              0.379057         32
  temp_ambient_value              0.379057         32

 Períodos críticos detectados: 2 (umbral fijo 50%)
                   inicio                       fin  n_steps  pct_faltantes_prom  pct_faltantes_max
2024-09-17 21:00:00+00:00 2024-09-19 05:00:00

In [15]:
def tratar_valores_faltantes_transformador(df, stats_categorias):
    """
    Trata valores faltantes con estrategias específicas por tipo de variable
    
    - Térmica: interpolación spline (suave)
    - Eléctrica: interpolación lineal (time si hay DatetimeIndex)
    - Mecánica: forward-fill (discretos)
    - Otras: lineal
    """
    print("\n Aplicando tratamiento especializado para valores faltantes...")
    
    df_treated = df.copy()
    missing_antes = df_treated.isnull().sum().sum()
    
    if missing_antes == 0:
        print(" No hay valores faltantes que tratar")
        return df_treated, 0, 0
    
    estrategias = {
        'Térmica':   {'método': 'spline', 'orden': 2, 'descripción': 'interpolación suave (cambios graduales)'},
        'Eléctrica': {'método': 'linear', 'descripción': 'interpolación lineal (cambios moderados)'},
        'Mecánica':  {'método': 'ffill',  'descripción': 'forward-fill (valores discretos)'},
        'Otras':     {'método': 'linear', 'descripción': 'interpolación lineal general'}
    }
    
    categorias_variables = {
        'Térmica':   ['temp', 'oil', 'hot', 'ambient', 'bubbling'],
        'Eléctrica': ['current', 'voltage', 'power', 'apparent'],
        'Mecánica':  ['position', 'tap'],
        'Otras':     []
    }
    
    valores_interpolados_por_categoria = {}
    
    for categoria, terminos in categorias_variables.items():
        # columnas por token
        columnas_categoria = [col for col in df_treated.columns 
                              if any(termino in col.lower() for termino in terminos)]
        
        # completar 'Otras'
        if not columnas_categoria and categoria == 'Otras':
            ya_categ = []
            for otros_terminos in [t for cat, t in categorias_variables.items() if cat != 'Otras']:
                for col in df_treated.columns:
                    if any(termino in col.lower() for termino in otros_terminos):
                        ya_categ.append(col)
            columnas_categoria = [col for col in df_treated.columns if col not in ya_categ]
        
        if not columnas_categoria:
            continue

        # pequeño filtro defensivo: solo columnas numéricas
        columnas_categoria = [c for c in columnas_categoria if pd.api.types.is_numeric_dtype(df_treated[c])]
        if not columnas_categoria:
            continue
            
        estrategia = estrategias[categoria]
        missing_antes_categoria = df_treated[columnas_categoria].isnull().sum().sum()
        
        print(f"\n Procesando {categoria}: {len(columnas_categoria)} variables")
        print(f"    Estrategia: {estrategia['descripción']}")
        
        try:
            if estrategia['método'] == 'spline' and isinstance(df_treated.index, pd.DatetimeIndex):
                # Interpolación spline para variables térmicas
                for col in columnas_categoria:
                    if df_treated[col].isnull().any():
                        df_treated[col] = df_treated[col].interpolate(
                            method='spline',
                            order=estrategia.get('orden', 2),
                            limit_direction='both'
                        )
                        
            elif estrategia['método'] == 'linear':
                # Interpolación lineal (time si hay DatetimeIndex)
                if isinstance(df_treated.index, pd.DatetimeIndex):
                    df_treated[columnas_categoria] = df_treated[columnas_categoria].interpolate(
                        method='time', limit_direction='both'
                    )
                else:
                    df_treated[columnas_categoria] = df_treated[columnas_categoria].interpolate(
                        method='linear', limit_direction='both'
                    )
                    
            elif estrategia['método'] == 'ffill':
                # Forward-fill y luego backward-fill
                df_treated[columnas_categoria] = df_treated[columnas_categoria].fillna(method='ffill')
                df_treated[columnas_categoria] = df_treated[columnas_categoria].fillna(method='bfill')
            
            # Éxito por categoría
            missing_despues_categoria = df_treated[columnas_categoria].isnull().sum().sum()
            interpolados_categoria = missing_antes_categoria - missing_despues_categoria
            valores_interpolados_por_categoria[categoria] = interpolados_categoria
            
            if missing_antes_categoria > 0:
                tasa_exito = (interpolados_categoria / missing_antes_categoria * 100)
                print(f"    Interpolados: {interpolados_categoria:,} valores ({tasa_exito:.1f}%)")
            
        except Exception as e:
            print(f"    Error en {categoria}: {str(e)}")
    
    # Resumen
    missing_despues = df_treated.isnull().sum().sum()
    total_interpolados = missing_antes - missing_despues
    
    print(f"\n Resumen del tratamiento:")
    print(f"    Valores faltantes antes: {missing_antes:,}")
    print(f"    Valores faltantes después: {missing_despues:,}")
    print(f"    Total interpolados: {total_interpolados:,}")
    
    if missing_antes > 0:
        tasa_exito_total = (total_interpolados / missing_antes * 100)
        print(f"    Tasa de éxito total: {tasa_exito_total:.1f}%")
    
    print(f"\n Desglose por categoría:")
    for categoria, interpolados in valores_interpolados_por_categoria.items():
        if interpolados > 0:
            print(f"    {categoria}: {interpolados:,} valores")
    
    return df_treated, missing_antes, total_interpolados

# Ejecutar tratamiento de valores faltantes
df_no_missing, missing_original, interpolated = tratar_valores_faltantes_transformador(
    df_typed, stats_faltantes
)



 Aplicando tratamiento especializado para valores faltantes...

 Procesando Térmica: 5 variables
    Estrategia: interpolación suave (cambios graduales)
    Interpolados: 167 valores (100.0%)

 Procesando Eléctrica: 3 variables
    Estrategia: interpolación lineal (cambios moderados)
    Interpolados: 121 valores (100.0%)

 Procesando Mecánica: 1 variables
    Estrategia: forward-fill (valores discretos)
    Interpolados: 40 valores (100.0%)

 Resumen del tratamiento:
    Valores faltantes antes: 328
    Valores faltantes después: 0
    Total interpolados: 328
    Tasa de éxito total: 100.0%

 Desglose por categoría:
    Térmica: 167 valores
    Eléctrica: 121 valores
    Mecánica: 40 valores


In [19]:
def definir_criterios_transformador():
    """
    Criterios técnicos para clasificación de estados (usando nombres normalizados):
      - temp_oil_value               (temperatura del aceite)
      - temp_spot_hot_value          (punto caliente)
      - temp_ambient_value           (ambiente)
      - current_load_value           (corriente de carga)
      - factor_carga                 (p.u., si lo calculas luego)
      - tap_position_value           (posición de tap)
    """
    criterios = {
        # Temperaturas (°C)
        'temp_oil_value': {
            'normal':  {'min': 35.8,  'max': 65.3},
            'alerta':  {'min': 65.3,  'max': 75.0},
            'critico': {'min': 75.0,  'max': float('inf')},
            'descripcion': 'Temperatura del aceite dieléctrico'
        },
        'temp_spot_hot_value': {
            'normal':  {'min': 38.7,  'max': 76.2},
            'alerta':  {'min': 76.2,  'max': 100.0},
            'critico': {'min': 100.0, 'max': float('inf')},
            'descripcion': 'Temperatura del punto caliente (hot-spot)'
        },
        'temp_ambient_value': {
            'normal':  {'min': -10.0, 'max': 40.0},
            'alerta':  {'min': 40.0,  'max': 50.0},
            'critico': {'min': 50.0,  'max': float('inf')},
            'descripcion': 'Temperatura ambiente'
        },

        # Eléctricas
        'current_load_value': {
            'normal':  {'min': 800.0,  'max': 2000.0},     # ~40–100% nominal   
            'alerta':  {'min': 2000.0, 'max': 2200.0},     # 100–110%
            'critico': {'min': 2200.0, 'max': float('inf')},
            'descripcion': 'Corriente de carga (A) lado BT'
        },
        'factor_carga': {
            'normal':  {'min': 0.0,  'max': 1.0},
            'alerta':  {'min': 1.0,  'max': 1.2},
            'critico': {'min': 1.2,  'max': float('inf')},
            'descripcion': 'Factor de carga (p.u.)'
        },

        # Mecánicas
        'tap_position_value': {
            'normal':  {'min': 6.0,  'max': 12.0},       # posición central ±3–4
            'alerta':  {'min': 3.0,  'max': 15.0},       # alejado del centro
            'critico': {'min': 0.0,  'max': 17.0},       # extremos
            'descripcion': 'Posición del tap OLTC (0–17)'
        }
    }

    # Criterios combinados (múltiples variables)
    criterios_combinados = {
        'sobrecarga_termica': {
            'variables': ['current_load_value', 'temp_oil_value'],
            'condicion': 'ambas en alerta o crítico',
            'severidad': 'critico',
            'descripcion': 'Sobrecarga con calentamiento excesivo'
        },
        'estres_termico': {
            'variables': ['temp_spot_hot_value', 'temp_oil_value'],
            'condicion': 'diferencia >25°C',
            'severidad': 'alerta',
            'descripcion': 'Gradiente térmico excesivo'
        },
        'regulacion_inestable': {
            'variables': ['tap_position_value'],
            'condicion': 'cambios frecuentes',
            'severidad': 'alerta',
            'descripcion': 'Regulación de voltaje inestable'
        }
    }

    return criterios, criterios_combinados

# Definir criterios técnicos
criterios_tecnicos, criterios_combinados = definir_criterios_transformador()

print(" Criterios técnicos definidos para transformadores:")
print(f"    Variables individuales: {len(criterios_tecnicos)}")
print(f"    Criterios combinados: {len(criterios_combinados)}")

print("\n Criterios principales:")
for variable, criterio in criterios_tecnicos.items():
    print(f"   • {variable}: {criterio['descripcion']}")
    print(f"     Normal: {criterio['normal']['min']}-{criterio['normal']['max']}")


 Criterios técnicos definidos para transformadores:
    Variables individuales: 6
    Criterios combinados: 3

 Criterios principales:
   • temp_oil_value: Temperatura del aceite dieléctrico
     Normal: 35.8-65.3
   • temp_spot_hot_value: Temperatura del punto caliente (hot-spot)
     Normal: 38.7-76.2
   • temp_ambient_value: Temperatura ambiente
     Normal: -10.0-40.0
   • current_load_value: Corriente de carga (A) lado BT
     Normal: 800.0-2000.0
   • factor_carga: Factor de carga (p.u.)
     Normal: 0.0-1.0
   • tap_position_value: Posición del tap OLTC (0–17)
     Normal: 6.0-12.0


In [21]:
def clasificar_estados_operacionales(df, criterios_tecnicos, criterios_combinados):
    """
    Clasifica estados operacionales del transformador con severidad graduada.
    Adaptado a nombres normalizados; sin añadir lógica innecesaria.
    """
    print(" Clasificando estados operacionales del transformador...")
    
    df_classified = df.copy()

    # Inicializar columnas de clasificación
    df_classified['estado_operacional'] = 'NORMAL'
    df_classified['nivel_severidad'] = 0  # 0: Normal, 1: Alerta, 2: Crítico
    df_classified['variables_anomalas'] = ''
    df_classified['descripcion_anomalia'] = ''

    contadores = {'NORMAL': 0, 'ALERTA': 0, 'CRITICO': 0}
    variables_evaluadas = 0

    print("\n Evaluando criterios individuales...")

    for criterio_nombre, criterio in criterios_tecnicos.items():
        # Usar coincidencia exacta (y permitir duplicados con sufijo)
        columnas_relacionadas = [c for c in df_classified.columns
                                 if c == criterio_nombre or c.startswith(criterio_nombre + "_")]

        if not columnas_relacionadas:
            continue

        variables_evaluadas += len(columnas_relacionadas)
        print(f"   {criterio_nombre}: {len(columnas_relacionadas)} columnas evaluadas")

        for col in columnas_relacionadas:
            if col not in df_classified.columns or not pd.api.types.is_numeric_dtype(df_classified[col]):
                continue

            valores = df_classified[col]

            mask_critico = (valores >= criterio['critico']['min']) & (valores < criterio['critico']['max'])
            mask_alerta  = (valores >= criterio['alerta']['min'])  & (valores < criterio['alerta']['max'])

            # CRÍTICO
            indices_critico = df_classified.index[mask_critico]
            df_classified.loc[indices_critico, 'estado_operacional'] = 'CRITICO'
            df_classified.loc[indices_critico, 'nivel_severidad'] = 2

            for idx in indices_critico:
                var_actuales = df_classified.loc[idx, 'variables_anomalas']
                df_classified.loc[idx, 'variables_anomalas'] = (var_actuales + ", " if var_actuales else "") + col

                desc_actual = df_classified.loc[idx, 'descripcion_anomalia']
                nueva_desc = f"{criterio['descripcion']} CRÍTICO"
                df_classified.loc[idx, 'descripcion_anomalia'] = (desc_actual + "; " if desc_actual else "") + nueva_desc

            # ALERTA (solo donde no sea ya crítico)
            indices_alerta = df_classified.index[mask_alerta & (df_classified['nivel_severidad'] < 2)]
            df_classified.loc[indices_alerta, 'estado_operacional'] = 'ALERTA'
            df_classified.loc[indices_alerta, 'nivel_severidad'] = 1

            for idx in indices_alerta:
                var_actuales = df_classified.loc[idx, 'variables_anomalas']
                df_classified.loc[idx, 'variables_anomalas'] = (var_actuales + ", " if var_actuales else "") + col

                desc_actual = df_classified.loc[idx, 'descripcion_anomalia']
                nueva_desc = f"{criterio['descripcion']} ALERTA"
                df_classified.loc[idx, 'descripcion_anomalia'] = (desc_actual + "; " if desc_actual else "") + nueva_desc

    print("\n Evaluando criterios combinados...")

    criterios_combinados_evaluados = 0

    # Sobrecarga térmica (corriente alta + temperatura aceite alta)
    corr_cols = [c for c in df_classified.columns if c.startswith('current_load_value')]
    oil_temp_cols = [c for c in df_classified.columns if c.startswith('temp_oil_value')]
    if not corr_cols:
        corr_cols = [c for c in df_classified.columns if 'current' in c.lower()]
    if not oil_temp_cols:
        oil_temp_cols = [c for c in df_classified.columns if ('oil' in c.lower() and 'temp' in c.lower())]

    if corr_cols and oil_temp_cols:
        for col_i in corr_cols:
            for col_t in oil_temp_cols:
                if pd.api.types.is_numeric_dtype(df_classified[col_i]) and pd.api.types.is_numeric_dtype(df_classified[col_t]):
                    mask_sobrecarga = (df_classified[col_i] > 2000) & (df_classified[col_t] > 70)
                    if mask_sobrecarga.any():
                        idxs = df_classified.index[mask_sobrecarga]
                        df_classified.loc[idxs, 'estado_operacional'] = 'CRITICO'
                        df_classified.loc[idxs, 'nivel_severidad'] = 2
                        for idx in idxs:
                            desc_actual = df_classified.loc[idx, 'descripcion_anomalia']
                            nueva_desc = "SOBRECARGA TÉRMICA COMBINADA"
                            if nueva_desc not in (desc_actual or ""):
                                df_classified.loc[idx, 'descripcion_anomalia'] = (desc_actual + "; " if desc_actual else "") + nueva_desc
                        criterios_combinados_evaluados += 1
                        print(f"   Sobrecarga térmica: {mask_sobrecarga.sum()} casos detectados")

    # Estrés térmico (hot-spot - aceite > 25°C)
    hot_cols = [c for c in df_classified.columns if c.startswith('temp_spot_hot_value')]
    if not hot_cols:
        hot_cols = [c for c in df_classified.columns if 'hot' in c.lower()]
    if oil_temp_cols and hot_cols:
        for col_hot in hot_cols:
            for col_oil in oil_temp_cols:
                if pd.api.types.is_numeric_dtype(df_classified[col_hot]) and pd.api.types.is_numeric_dtype(df_classified[col_oil]):
                    diferencia = df_classified[col_hot] - df_classified[col_oil]
                    mask_estres = diferencia > 25
                    if mask_estres.any():
                        idxs = df_classified.index[mask_estres & (df_classified['nivel_severidad'] < 2)]
                        if len(idxs) > 0:
                            df_classified.loc[idxs, 'estado_operacional'] = 'ALERTA'
                            df_classified.loc[idxs, 'nivel_severidad'] = 1
                            for idx in idxs:
                                desc_actual = df_classified.loc[idx, 'descripcion_anomalia']
                                nueva_desc = f"GRADIENTE TÉRMICO EXCESIVO ({diferencia.loc[idx]:.1f}°C)"
                                df_classified.loc[idx, 'descripcion_anomalia'] = (desc_actual + "; " if desc_actual else "") + nueva_desc
                            criterios_combinados_evaluados += 1
                            print(f"    Estrés térmico: {len(idxs)} casos detectados")

    # Resumen final
    contadores = df_classified['estado_operacional'].value_counts().to_dict()
    total_registros = len(df_classified)

    print("\n Resumen de clasificación:")
    print(f"    Total de registros: {total_registros:,}")
    print(f"    Variables evaluadas: {variables_evaluadas}")
    print(f"    Criterios combinados evaluados: {criterios_combinados_evaluados}")

    print("\n Distribución de estados:")
    for estado in ['NORMAL', 'ALERTA', 'CRITICO']:
        cantidad = contadores.get(estado, 0)
        porcentaje = (cantidad / total_registros * 100) if total_registros > 0 else 0
        emoji = {'NORMAL': '✅', 'ALERTA': '⚠️', 'CRITICO': '🚨'}[estado]
        print(f"   {emoji} {estado}: {cantidad:,} registros ({porcentaje:.1f}%)")

    return df_classified, contadores

# Ejecutar clasificación de estados
df_classified, estadisticas_estados = clasificar_estados_operacionales(
    df_no_missing, criterios_tecnicos, criterios_combinados
)


 Clasificando estados operacionales del transformador...

 Evaluando criterios individuales...
   temp_oil_value: 1 columnas evaluadas
   temp_spot_hot_value: 1 columnas evaluadas
   temp_ambient_value: 1 columnas evaluadas
   current_load_value: 1 columnas evaluadas
   tap_position_value: 1 columnas evaluadas

 Evaluando criterios combinados...

 Resumen de clasificación:
    Total de registros: 8,442
    Variables evaluadas: 5
    Criterios combinados evaluados: 0

 Distribución de estados:
   ✅ NORMAL: 13 registros (0.2%)
   ⚠️ ALERTA: 0 registros (0.0%)
   🚨 CRITICO: 8,429 registros (99.8%)


In [22]:
def clasificar_estados_operacionales(df, criterios_tecnicos, criterios_combinados):
    """
    Clasifica estados operacionales del transformador con severidad graduada.
    Adaptado a nombres normalizados; sin añadir lógica innecesaria.
    """
    print(" Clasificando estados operacionales del transformador...")
    
    df_classified = df.copy()

    # Inicializar columnas de clasificación
    df_classified['estado_operacional'] = 'NORMAL'
    df_classified['nivel_severidad'] = 0  # 0: Normal, 1: Alerta, 2: Crítico
    df_classified['variables_anomalas'] = ''
    df_classified['descripcion_anomalia'] = ''

    contadores = {'NORMAL': 0, 'ALERTA': 0, 'CRITICO': 0}
    variables_evaluadas = 0

    print("\n Evaluando criterios individuales...")

    for criterio_nombre, criterio in criterios_tecnicos.items():
        # Usar coincidencia exacta (y permitir duplicados con sufijo)
        columnas_relacionadas = [c for c in df_classified.columns
                                 if c == criterio_nombre or c.startswith(criterio_nombre + "_")]

        if not columnas_relacionadas:
            continue

        variables_evaluadas += len(columnas_relacionadas)
        print(f"    {criterio_nombre}: {len(columnas_relacionadas)} columnas evaluadas")

        for col in columnas_relacionadas:
            if col not in df_classified.columns or not pd.api.types.is_numeric_dtype(df_classified[col]):
                continue

            valores = df_classified[col]

            mask_critico = (valores >= criterio['critico']['min']) & (valores < criterio['critico']['max'])
            mask_alerta  = (valores >= criterio['alerta']['min'])  & (valores < criterio['alerta']['max'])

            # CRÍTICO
            indices_critico = df_classified.index[mask_critico]
            df_classified.loc[indices_critico, 'estado_operacional'] = 'CRITICO'
            df_classified.loc[indices_critico, 'nivel_severidad'] = 2

            for idx in indices_critico:
                var_actuales = df_classified.loc[idx, 'variables_anomalas']
                df_classified.loc[idx, 'variables_anomalas'] = (var_actuales + ", " if var_actuales else "") + col

                desc_actual = df_classified.loc[idx, 'descripcion_anomalia']
                nueva_desc = f"{criterio['descripcion']} CRÍTICO"
                df_classified.loc[idx, 'descripcion_anomalia'] = (desc_actual + "; " if desc_actual else "") + nueva_desc

            # ALERTA (solo donde no sea ya crítico)
            indices_alerta = df_classified.index[mask_alerta & (df_classified['nivel_severidad'] < 2)]
            df_classified.loc[indices_alerta, 'estado_operacional'] = 'ALERTA'
            df_classified.loc[indices_alerta, 'nivel_severidad'] = 1

            for idx in indices_alerta:
                var_actuales = df_classified.loc[idx, 'variables_anomalas']
                df_classified.loc[idx, 'variables_anomalas'] = (var_actuales + ", " if var_actuales else "") + col

                desc_actual = df_classified.loc[idx, 'descripcion_anomalia']
                nueva_desc = f"{criterio['descripcion']} ALERTA"
                df_classified.loc[idx, 'descripcion_anomalia'] = (desc_actual + "; " if desc_actual else "") + nueva_desc

    print("\nEvaluando criterios combinados...")

    criterios_combinados_evaluados = 0

    # Sobrecarga térmica (corriente alta + temperatura aceite alta)
    corr_cols = [c for c in df_classified.columns if c.startswith('current_load_value')]
    oil_temp_cols = [c for c in df_classified.columns if c.startswith('temp_oil_value')]
    if not corr_cols:
        corr_cols = [c for c in df_classified.columns if 'current' in c.lower()]
    if not oil_temp_cols:
        oil_temp_cols = [c for c in df_classified.columns if ('oil' in c.lower() and 'temp' in c.lower())]

    if corr_cols and oil_temp_cols:
        for col_i in corr_cols:
            for col_t in oil_temp_cols:
                if pd.api.types.is_numeric_dtype(df_classified[col_i]) and pd.api.types.is_numeric_dtype(df_classified[col_t]):
                    mask_sobrecarga = (df_classified[col_i] > 2000) & (df_classified[col_t] > 70)
                    if mask_sobrecarga.any():
                        idxs = df_classified.index[mask_sobrecarga]
                        df_classified.loc[idxs, 'estado_operacional'] = 'CRITICO'
                        df_classified.loc[idxs, 'nivel_severidad'] = 2
                        for idx in idxs:
                            desc_actual = df_classified.loc[idx, 'descripcion_anomalia']
                            nueva_desc = "SOBRECARGA TÉRMICA COMBINADA"
                            if nueva_desc not in (desc_actual or ""):
                                df_classified.loc[idx, 'descripcion_anomalia'] = (desc_actual + "; " if desc_actual else "") + nueva_desc
                        criterios_combinados_evaluados += 1
                        print(f"    Sobrecarga térmica: {mask_sobrecarga.sum()} casos detectados")

    # Estrés térmico (hot-spot - aceite > 25°C)
    hot_cols = [c for c in df_classified.columns if c.startswith('temp_spot_hot_value')]
    if not hot_cols:
        hot_cols = [c for c in df_classified.columns if 'hot' in c.lower()]
    if oil_temp_cols and hot_cols:
        for col_hot in hot_cols:
            for col_oil in oil_temp_cols:
                if pd.api.types.is_numeric_dtype(df_classified[col_hot]) and pd.api.types.is_numeric_dtype(df_classified[col_oil]):
                    diferencia = df_classified[col_hot] - df_classified[col_oil]
                    mask_estres = diferencia > 25
                    if mask_estres.any():
                        idxs = df_classified.index[mask_estres & (df_classified['nivel_severidad'] < 2)]
                        if len(idxs) > 0:
                            df_classified.loc[idxs, 'estado_operacional'] = 'ALERTA'
                            df_classified.loc[idxs, 'nivel_severidad'] = 1
                            for idx in idxs:
                                desc_actual = df_classified.loc[idx, 'descripcion_anomalia']
                                nueva_desc = f"GRADIENTE TÉRMICO EXCESIVO ({diferencia.loc[idx]:.1f}°C)"
                                df_classified.loc[idx, 'descripcion_anomalia'] = (desc_actual + "; " if desc_actual else "") + nueva_desc
                            criterios_combinados_evaluados += 1
                            print(f"    Estrés térmico: {len(idxs)} casos detectados")

    # Resumen final
    contadores = df_classified['estado_operacional'].value_counts().to_dict()
    total_registros = len(df_classified)

    print("\n Resumen de clasificación:")
    print(f"    Total de registros: {total_registros:,}")
    print(f"    Variables evaluadas: {variables_evaluadas}")
    print(f"    Criterios combinados evaluados: {criterios_combinados_evaluados}")

    print("\nDistribución de estados:")
    for estado in ['NORMAL', 'ALERTA', 'CRITICO']:
        cantidad = contadores.get(estado, 0)
        porcentaje = (cantidad / total_registros * 100) if total_registros > 0 else 0
        emoji = {'NORMAL': '✅', 'ALERTA': '⚠️', 'CRITICO': '🚨'}[estado]
        print(f"   {emoji} {estado}: {cantidad:,} registros ({porcentaje:.1f}%)")

    return df_classified, contadores

# Ejecutar clasificación de estados
df_classified, estadisticas_estados = clasificar_estados_operacionales(
    df_no_missing, criterios_tecnicos, criterios_combinados
)


 Clasificando estados operacionales del transformador...

 Evaluando criterios individuales...
    temp_oil_value: 1 columnas evaluadas
    temp_spot_hot_value: 1 columnas evaluadas
    temp_ambient_value: 1 columnas evaluadas
    current_load_value: 1 columnas evaluadas
    tap_position_value: 1 columnas evaluadas

Evaluando criterios combinados...

 Resumen de clasificación:
    Total de registros: 8,442
    Variables evaluadas: 5
    Criterios combinados evaluados: 0

Distribución de estados:
   ✅ NORMAL: 13 registros (0.2%)
   ⚠️ ALERTA: 0 registros (0.0%)
   🚨 CRITICO: 8,429 registros (99.8%)


In [23]:
def generar_reporte_calidad_transformador(df_original, df_final, missing_original, interpolated, estadisticas_estados):
    """
    Genera reporte completo de calidad para transformadores eléctricos.
    Adaptado a nombres normalizados (temp_oil_value, temp_spot_hot_value, current_load_value, etc.).
    """
    print("REPORTE DE CALIDAD - TRANSFORMADORES ELÉCTRICOS")
    print("=" * 70)

    # 1) Información general
    print("\n 1. INFORMACIÓN DEL TRANSFORMADOR")
    print(f"    Registros temporales: {len(df_final):,}")
    print(f"    Variables monitoreadas: {len(df_final.columns)}")
    if isinstance(df_final.index, pd.DatetimeIndex) and len(df_final) > 0:
        print(f"    Período de monitoreo: {df_final.index.min()} a {df_final.index.max()}")
        print(f"   ⏱ Duración total: {df_final.index.max() - df_final.index.min()}")

    # 2) Calidad de datos
    print("\n2. CALIDAD DE DATOS TÉCNICOS")
    missing_final = int(df_final.isnull().sum().sum())
    total_valores = int(df_final.size) if df_final.size else 1  # evitar /0
    print(f"   Valores faltantes originales: {missing_original:,}")
    print(f"   Valores interpolados: {interpolated:,}")
    print(f"   Valores faltantes finales: {missing_final:,}")
    print(f"   Completitud de datos: {((total_valores - missing_final) / total_valores * 100):.2f}%")

    # 3) Análisis por categoría técnica (tokens normalizados)
    print("\n 3. ANÁLISIS POR CATEGORÍA TÉCNICA")
    categorias = {
        'Térmica':   ['temp', 'oil', 'hot', 'ambient'],
        'Eléctrica': ['current', 'voltage', 'power', 'apparent'],
        'Mecánica':  ['position', 'tap'],
    }

    for categoria, terminos in categorias.items():
        cols_categoria = [c for c in df_final.columns if any(t in c.lower() for t in terminos)]
        if cols_categoria:
            datos_categoria = df_final[cols_categoria]
            total_cat = datos_categoria.size if datos_categoria.size else 1
            completitud = ((total_cat - datos_categoria.isnull().sum().sum()) / total_cat * 100)
            print(f"    {categoria}: {len(cols_categoria)} variables, completitud {completitud:.1f}%")

            if categoria == 'Térmica':
                temp_cols = [c for c in cols_categoria if 'temp' in c.lower()]
                if temp_cols:
                    temp_mean = df_final[temp_cols].mean().mean()
                    print(f"       Temperatura promedio: {temp_mean:.1f}°C")
            elif categoria == 'Eléctrica':
                current_cols = [c for c in cols_categoria if 'current' in c.lower()]
                if current_cols:
                    current_mean = df_final[current_cols].mean().mean()
                    print(f"       Corriente promedio: {current_mean:.0f}A")

    # 4) Estados operacionales
    print("\n4. ESTADOS OPERACIONALES")
    total_estados = sum(estadisticas_estados.values()) if estadisticas_estados else 0
    for estado, cantidad in estadisticas_estados.items():
        porcentaje = (cantidad / total_estados * 100) if total_estados > 0 else 0
        emoji = {'NORMAL': '✅', 'ALERTA': '⚠️', 'CRITICO': '🚨'}.get(estado, '📊')
        print(f"   {emoji} {estado}: {cantidad:,} registros ({porcentaje:.1f}%)")

    porcentaje_normal = (estadisticas_estados.get('NORMAL', 0) / total_estados * 100) if total_estados > 0 else 0
    porcentaje_critico = (estadisticas_estados.get('CRITICO', 0) / total_estados * 100) if total_estados > 0 else 0

    print(f"\nEvaluación de la distribución:")
    if porcentaje_normal > 80:
        print(f"    Distribución saludable: {porcentaje_normal:.1f}% operación normal")
    elif porcentaje_normal > 60:
        print(f"    Distribución moderada: {porcentaje_normal:.1f}% operación normal")
    else:
        print(f"    Distribución preocupante: solo {porcentaje_normal:.1f}% operación normal")

    if porcentaje_critico > 10:
        print(f"    Alto porcentaje crítico: {porcentaje_critico:.1f}% requiere atención")
    elif porcentaje_critico > 5:
        print(f"   Porcentaje crítico moderado: {porcentaje_critico:.1f}%")
    else:
        print(f"   Bajo porcentaje crítico: {porcentaje_critico:.1f}%")

    # 5) Validaciones de coherencia física (hot-spot >= oil)
    print("\n 5. VALIDACIONES DE COHERENCIA FÍSICA")
    oil_cols = [c for c in df_final.columns if ('oil' in c.lower() and 'temp' in c.lower())]
    hot_cols = [c for c in df_final.columns if ('hot' in c.lower() and 'temp' in c.lower())]

    if oil_cols and hot_cols:
        oil_data = df_final[oil_cols[0]].dropna()
        hot_data = df_final[hot_cols[0]].dropna()
        idx_comunes = oil_data.index.intersection(hot_data.index)
        if len(idx_comunes) > 0:
            coherentes = (hot_data.loc[idx_comunes] >= oil_data.loc[idx_comunes]).sum()
            total_comp = len(idx_comunes)
            coherencia_termica = (coherentes / total_comp * 100) if total_comp else 0.0
            print(f"    Coherencia térmica: {coherencia_termica:.1f}% (punto caliente ≥ aceite)")
            if coherencia_termica > 90:
                print("      Coherencia excelente")
            elif coherencia_termica > 70:
                print("      Coherencia moderada")
            else:
                print("      Coherencia baja - revisar calibración de sensores")

    # 6) Recomendaciones
    print("\n 6. RECOMENDACIONES TÉCNICAS")
    recomendaciones = []
    if total_valores > 0 and missing_final > total_valores * 0.05:
        recomendaciones.append("Revisar sistema de adquisición de datos - alta proporción de valores faltantes")
    if porcentaje_critico > 15:
        recomendaciones.append("Evaluar estado del transformador - alto porcentaje de condiciones críticas")
    if len(df_final.columns) < 5:
        recomendaciones.append("Considerar ampliar monitoreo - pocas variables disponibles")
    if not recomendaciones:
        recomendaciones.append("Dataset en condiciones óptimas para análisis predictivo")
    for i, rec in enumerate(recomendaciones, 1):
        print(f"   {i}. {rec}")

    print("\n" + "=" * 70)
    print(" PREPROCESAMIENTO DE TRANSFORMADOR COMPLETADO EXITOSAMENTE")
    print(" Dataset listo para análisis de mantenimiento predictivo")
    print("=" * 70)

    # Salida (igual estructura que tu versión)
    return {
        'completitud_total': ((total_valores - missing_final) / total_valores * 100) if total_valores else 0.0,
        'estados_operacionales': estadisticas_estados,
        'variables_por_categoria': {
            cat: len([c for c in df_final.columns if any(t in c.lower() for t in terminos)])
            for cat, terminos in categorias.items()
        },
        'recomendaciones': recomendaciones
    }

# Generar reporte de calidad completo
metricas_calidad = generar_reporte_calidad_transformador(
    datos_transformador,  # no se usa dentro; lo dejamos por compatibilidad
    df_classified,
    missing_original,
    interpolated,
    estadisticas_estados
)


REPORTE DE CALIDAD - TRANSFORMADORES ELÉCTRICOS

 1. INFORMACIÓN DEL TRANSFORMADOR
    Registros temporales: 8,442
    Variables monitoreadas: 13
    Período de monitoreo: 2024-09-10 04:00:00+00:00 a 2025-08-27 21:00:00+00:00
   ⏱ Duración total: 351 days 17:00:00

2. CALIDAD DE DATOS TÉCNICOS
   Valores faltantes originales: 328
   Valores interpolados: 328
   Valores faltantes finales: 0
   Completitud de datos: 100.00%

 3. ANÁLISIS POR CATEGORÍA TÉCNICA
    Térmica: 5 variables, completitud 100.0%
       Temperatura promedio: 66.6°C
    Eléctrica: 3 variables, completitud 100.0%
       Corriente promedio: 690A
    Mecánica: 1 variables, completitud 100.0%

4. ESTADOS OPERACIONALES
   🚨 CRITICO: 8,429 registros (99.8%)
   ✅ NORMAL: 13 registros (0.2%)

Evaluación de la distribución:
    Distribución preocupante: solo 0.2% operación normal
    Alto porcentaje crítico: 99.8% requiere atención

 5. VALIDACIONES DE COHERENCIA FÍSICA
    Coherencia térmica: 99.9% (punto caliente ≥ aceite

In [27]:
from pathlib import Path
from datetime import datetime
import pandas as pd
from deltalake import write_deltalake

def guardar_dataset_transformador(
    df: pd.DataFrame,
    ruta_destino: Path,
    metricas_calidad: dict,
    criterios_tecnicos: dict,
    silver_table_path: Path,
    mode: str = "overwrite",            # "overwrite" o "append"
    exportar_csv_parquet: bool = False  # exportables sueltos (además de Delta)
):
    """
    Guarda el dataset procesado:
      1) Escribe tabla Silver en Delta Lake (particionada por year/month).
      2) (Opcional) Exporta CSV/Parquet "sueltos" a ruta_destino.
      3) Genera metadatos técnicos y resumen de variables en ruta_destino.
    """
    ruta_destino = Path(ruta_destino)
    ruta_destino.mkdir(parents=True, exist_ok=True)

    silver_table_path = Path(silver_table_path)
    silver_table_path.mkdir(parents=True, exist_ok=True)

    print(" Guardando dataset de transformador...")
    print(f"    Dimensiones (entrada): {df.shape[0]:,} × {df.shape[1]}")

    resultados = {}

    # ================== 1) SILVER en DELTA ==================
    try:
        # Normalizar timestamp para Delta
        out = df.copy()
        if isinstance(out.index, pd.DatetimeIndex):
            out = out.reset_index().rename(columns={"index": "timestamp"})

        if "timestamp" not in out.columns:
            raise ValueError("No se encontró columna 'timestamp' para escribir la tabla Silver.")

        out["timestamp"] = pd.to_datetime(out["timestamp"], errors="coerce", utc=True)

        # Particiones
        out["year"]  = out["timestamp"].dt.year.astype("int32")
        out["month"] = out["timestamp"].dt.month.astype("int8")

        # Escribir tabla Delta (sin overwrite_schema)
        write_deltalake(
            str(silver_table_path),
            out,
            mode=mode,                           # "overwrite" o "append"
            partition_by=["year", "month"]
        )
        print(f" Silver (Delta) escrita en: {silver_table_path}")
        print("    Particiones: year / month")
        resultados["silver_delta"] = {"exito": True, "path": str(silver_table_path), "mode": mode}
    except Exception as e:
        print(f"Error al escribir Silver (Delta): {e}")
        resultados["silver_delta"] = {"exito": False, "error": str(e)}
        # Si falla la escritura de la tabla principal, devolvemos resultados
        return resultados

    # ================== 2) EXPORTABLES OPCIONALES ==================
    if exportar_csv_parquet:
        # CSV principal
        try:
            archivo_csv = ruta_destino / "transformador_data.csv"
            out.to_csv(archivo_csv, index=False, encoding="utf-8")
            size_mb = archivo_csv.stat().st_size / (1024 * 1024)
            print(f"   CSV principal: {archivo_csv.name} ({size_mb:.1f} MB)")
            resultados["dataset_principal_csv"] = {"exito": True, "tamaño_mb": size_mb}
        except Exception as e:
            print(f"   Error al exportar CSV: {e}")
            resultados["dataset_principal_csv"] = {"exito": False, "error": str(e)}

        # Parquet principal
        try:
            archivo_parquet = ruta_destino / "transformador_data.parquet"
            out.to_parquet(archivo_parquet, engine="pyarrow", index=False)
            size_mb = archivo_parquet.stat().st_size / (1024 * 1024)
            print(f"    Parquet principal: {archivo_parquet.name} ({size_mb:.1f} MB)")
            resultados["dataset_principal_parquet"] = {"exito": True, "tamaño_mb": size_mb}
        except Exception as e:
            print(f"    Error al exportar Parquet: {e}")
            resultados["dataset_principal_parquet"] = {"exito": False, "error": str(e)}

        # Dataset de anomalías (si existe la etiqueta)
        try:
            if "estado_operacional" in df.columns:
                df_anomalias = df[df["estado_operacional"].isin(["ALERTA", "CRITICO"])].copy()
                if not df_anomalias.empty:
                    # CSV
                    archivo_anom_csv = ruta_destino / "transformador_data_anomalias.csv"
                    df_anomalias.to_csv(archivo_anom_csv, index=True, encoding="utf-8")
                    # Parquet
                    archivo_anom_parquet = ruta_destino / "transformador_data_anomalias.parquet"
                    df_anomalias.to_parquet(archivo_anom_parquet, engine="pyarrow", index=False)
                    print(f"    Anomalías exportadas: {len(df_anomalias):,} registros")
                    resultados["anomalias_export"] = {
                        "exito": True,
                        "registros": int(len(df_anomalias))
                    }
                else:
                    print("    Sin anomalías para exportar")
                    resultados["anomalias_export"] = {"exito": True, "registros": 0}
        except Exception as e:
            print(f"   Error al exportar anomalías: {e}")
            resultados["anomalias_export"] = {"exito": False, "error": str(e)}

    # ================== 3) METADATOS TÉCNICOS ==================
    try:
        archivo_metadatos = ruta_destino / "transformador_data_metadatos_tecnicos.txt"
        with open(archivo_metadatos, "w", encoding="utf-8") as f:
            f.write("METADATOS TÉCNICOS - TRANSFORMADOR ELÉCTRICO\n")
            f.write("=" * 50 + "\n\n")

            # Info temporal (usar 'out' con timestamp seguro)
            periodo_min = out["timestamp"].min()
            periodo_max = out["timestamp"].max()

            f.write(f"Timestamp de procesamiento: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write("Tipo de equipo: Transformador de Potencia\n")
            f.write(f"Período de datos: {periodo_min} a {periodo_max}\n")
            f.write(f"Duración del monitoreo: {periodo_max - periodo_min}\n")
            f.write("Resolución temporal: 1 hora\n")
            f.write(f"Total de registros: {len(out):,}\n")
            f.write(f"Total de variables: {df.shape[1]}\n\n")

            f.write("CRITERIOS DE CLASIFICACIÓN OPERACIONAL\n")
            f.write("-" * 40 + "\n\n")
            for nombre, criterio in criterios_tecnicos.items():
                f.write(f"{nombre.upper()}:\n")
                f.write(f"  Descripción: {criterio['descripcion']}\n")
                f.write(f"  Normal: {criterio['normal']['min']} - {criterio['normal']['max']}\n")
                f.write(f"  Alerta: {criterio['alerta']['min']} - {criterio['alerta']['max']}\n")
                f.write(f"  Crítico: {criterio['critico']['min']} - {criterio['critico']['max']}\n\n")

            f.write("DISTRIBUCIÓN DE ESTADOS OPERACIONALES\n")
            f.write("-" * 40 + "\n\n")
            for estado, cantidad in metricas_calidad["estados_operacionales"].items():
                porcentaje = (cantidad / len(df) * 100) if len(df) > 0 else 0
                f.write(f"{estado}: {cantidad:,} registros ({porcentaje:.1f}%)\n")

            f.write(f"\nCompletitud de datos: {metricas_calidad['completitud_total']:.2f}%\n")

            f.write("\nVARIABLES POR CATEGORÍA TÉCNICA\n")
            f.write("-" * 40 + "\n\n")
            for categoria, cantidad in metricas_calidad["variables_por_categoria"].items():
                f.write(f"{categoria}: {cantidad} variables\n")

        # print(f"    Metadatos técnicos: {archivo_metadatos.name}")
        # resultados["metadatos_tecnicos"] = {"exito": True, "path": str(archivo_metadatos)}
    except Exception as e:
        print(f"    Error en metadatos técnicos: {e}")
        resultados["metadatos_tecnicos"] = {"exito": False, "error": str(e)}

    # ================== 4) RESUMEN DE VARIABLES ==================
    try:
        archivo_vars = ruta_destino / "transformador_data_variables.csv"
        variables_info = []

        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                serie = df[col].dropna()
                if len(serie) > 0:
                    info = {
                        "variable": col,
                        "tipo": "numérica",
                        "valores_validos": int(len(serie)),
                        "valores_faltantes": int(df[col].isnull().sum()),
                        "porcentaje_completitud": float(len(serie) / len(df) * 100),
                        "minimo": float(serie.min()),
                        "maximo": float(serie.max()),
                        "media": float(serie.mean()),
                        "mediana": float(serie.median()),
                        "desviacion_estandar": float(serie.std())
                    }
                else:
                    info = {
                        "variable": col,
                        "tipo": "numérica",
                        "valores_validos": 0,
                        "valores_faltantes": int(df[col].isnull().sum()),
                        "porcentaje_completitud": 0.0,
                        "minimo": None,
                        "maximo": None,
                        "media": None,
                        "mediana": None,
                        "desviacion_estandar": None
                    }

                # Etiqueta por categoría técnica
                cl = col.lower()
                if any(t in cl for t in ["temp", "oil", "hot", "ambient"]):
                    info["categoria_tecnica"] = "Térmica"
                elif any(t in cl for t in ["current", "voltage", "power", "apparent"]):
                    info["categoria_tecnica"] = "Eléctrica"
                elif any(t in cl for t in ["position", "tap"]):
                    info["categoria_tecnica"] = "Mecánica"
                else:
                    info["categoria_tecnica"] = "Otra"

                variables_info.append(info)
            else:
                info = {
                    "variable": col,
                    "tipo": "categórica",
                    "valores_validos": int(df[col].notna().sum()),
                    "valores_faltantes": int(df[col].isnull().sum()),
                    "porcentaje_completitud": float(df[col].notna().sum() / len(df) * 100),
                    "valores_unicos": int(df[col].nunique()),
                    "categoria_tecnica": "Clasificación"
                }
                variables_info.append(info)

        pd.DataFrame(variables_info).to_csv(archivo_vars, index=False, encoding="utf-8")
        print(f"   Resumen de variables: {len(variables_info)} variables")
        resultados["resumen_variables"] = {"exito": True, "variables_documentadas": len(variables_info)}
    except Exception as e:
        print(f"   Error en resumen de variables: {e}")
        resultados["resumen_variables"] = {"exito": False, "error": str(e)}

    # ================== 5) RESUMEN FINAL ==================
    # ok = sum(1 for r in resultados.values() if r.get("exito"))
    # print("\n Resumen del guardado:")
    # print(f"    Éxitos: {ok}/{len(resultados)}")
    # print(f"    Silver (Delta): {silver_table_path}")

    return resultados


# ===== Ejecución (usando tus rutas ya definidas) =====
resultado_guardado = guardar_dataset_transformador(
    df_classified,
    ruta_destino=RUTA_PROCESSED,       # artefactos (metadatos, resúmenes, exportables)
    metricas_calidad=metricas_calidad,
    criterios_tecnicos=criterios_tecnicos,
    silver_table_path=RUTA_SILVER,     # tabla Silver en Delta
    mode="overwrite",                  # usa "append" para cargas incrementales
    exportar_csv_parquet=False         # True si quieres CSV/Parquet adicionales
)


 Guardando dataset de transformador...
    Dimensiones (entrada): 8,442 × 13
 Silver (Delta) escrita en: C:\Users\Asus TUF\Desktop\proy_ml\data\capa_silver\preprocesamiento_silver
    Particiones: year / month
   Resumen de variables: 13 variables


In [26]:
df_classified.head(5)

,current_load_value,power_apparent_value,tap_position_value,temp_oil_value,temp_oil_oltc_value,temp_ambient_value,temp_bubbling_value,temp_spot_hot_value,voltage_value,estado_operacional,nivel_severidad,variables_anomalas,descripcion_anomalia
2024-09-10 04:00:00+00:00,721.116618,29.967213,11,53.500000,53.561257,26.675000,178.692130,58.642727,130.860557,CRITICO,2,tap_position_value,Posición del tap OLTC (0–17) CRÍTICO
2024-09-10 05:00:00+00:00,675.654028,28.100183,11,52.950000,51.526277,25.900000,176.250397,57.149869,130.920054,CRITICO,2,tap_position_value,Posición del tap OLTC (0–17) CRÍTICO
2024-09-10 06:00:00+00:00,639.430143,26.682619,11,52.400000,49.400002,24.866667,175.504532,56.585251,131.428114,CRITICO,2,tap_position_value,Posición del tap OLTC (0–17) CRÍTICO
2024-09-10 07:00:00+00:00,603.393705,25.287238,11,49.900002,48.700001,20.500000,174.745842,54.969161,131.961710,CRITICO,2,tap_position_value,Posición del tap OLTC (0–17) CRÍTICO
2024-09-10 08:00:00+00:00,593.142869,24.792122,11,49.300001,47.716667,19.925000,173.987152,53.353071,131.792326,CRITICO,2,tap_position_value,Posición del tap OLTC (0–17) CRÍTICO
